# Commitment signature explorer

Pick a level (lineage / cell_type) and a label pair A, B. B can be left blank
for the complement (A vs not-A). Puts the repo root on sys.path; needs the index built
once via `python -m analysis.pipelines.commitment.build_index`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))  # repo root on path
%load_ext autoreload
%autoreload 2
from analysis.pipelines.commitment import commitment as C
idx = C.Index()
len(idx.nodes), idx.expr.shape

## One query

In [ ]:
res = C.run(idx, 'lineage', 'neural ectoderm', 'neural crest',
            cutoff=0.4, max_per_group=400)
fig = C.plot(res)

## Interactive
Dropdowns for level / A / B / cutoff. B = '(complement)' means A vs not-A.

In [ ]:
import ipywidgets as w
from IPython.display import display
import matplotlib.pyplot as plt

level = w.Dropdown(options=C.LEVELS, value='lineage', description='level')
a = w.Dropdown(description='A')
b = w.Dropdown(description='B')
cutoff = w.FloatSlider(value=0.4, min=0.1, max=0.9, step=0.05, description='cutoff')
out = w.Output()

def set_labels(*_):
    opts = idx.cols[level.value]
    a.options = opts
    b.options = ['(complement)'] + opts
level.observe(set_labels, 'value'); set_labels()

def go(_):
    with out:
        out.clear_output(wait=True)
        bb = None if b.value == '(complement)' else b.value
        res = C.run(idx, level.value, a.value, bb, cutoff=cutoff.value,
                    max_per_group=400)
        C.plot(res); plt.show()

run_btn = w.Button(description='run', button_style='primary')
run_btn.on_click(go)
display(w.VBox([level, a, b, cutoff, run_btn, out]))

## Global commitment (derived view)
Each node's single dominant label at a cutoff, independent of any pair.

In [ ]:
import pandas as pd
gc = C.global_commitment(idx, 'lineage', 0.5)
pd.Series(gc).value_counts()